# Week 3, day 2 — Worksheet 08 SOLUTIONS: binning   (L05)

Executed in the lab image (pandas 3.0.5) against the real
`data/orders_long.csv`. Every quoted number is what it actually printed.

Questions 3 and 4 are the ones to re-read. Values outside your edges are
discarded silently, and the deck never mentions it.

Run this cell once to set up the data. Then work down the sheet.

In [ ]:
# Worksheet 08 — Binning. Run this once.
import pandas as pd

orders = pd.read_csv("data/orders_long.csv")

print("Sales range: %.2f to %.2f" % (orders["Sales"].min(), orders["Sales"].max()))
print("Quantity range: %d to %d" % (orders["Quantity"].min(), orders["Quantity"].max()))

PART A — cut: edges you choose

### Question 1

`5 -> Child`, `17 -> Teen`, `22 -> Young Adult`, **`35 -> Young Adult`**, `45 -> Adult`, `67 -> Senior`, `90 -> Senior`. -> `6` edges, `5` labels.

Five bins need six edges, because each bin is defined by the pair around
it. Get that off by one and `cut` raises rather than guessing, which is the
one place in this sheet it protects you.

Age 35 is the interesting one — see Q2.

In [ ]:
ages = [5, 17, 22, 35, 45, 67, 90]
bins = [0, 12, 19, 35, 60, 100]
labels = ["Child", "Teen", "Young Adult", "Adult", "Senior"]
out = pd.cut(ages, bins=bins, labels=labels)

for a, l in zip(ages, out):
    print("  %3d -> %s" % (a, l))
print()
print("edges:", len(bins), "labels:", len(labels))

### Question 2

Intervals print as `(0, 12] < (12, 19] < (19, 35] < (35, 60] < (60, 100]`. -> age 35 lands in **`(19, 35]`**, the *lower* bin.

Intervals are **right-closed** by default: the right edge belongs to the
bin, the left edge does not. So 35 is a Young Adult and 36 is an Adult.

That is a real decision about a real boundary, and it is a default rather
than something you chose. `right=False` flips it to `[19, 35)`, making 35 an
Adult instead.

On ages it feels arbitrary. On a price threshold, a grade boundary or an
SLA it is a policy — 'orders up to £500' and 'orders under £500' differ by
every order costing exactly £500.

In [ ]:
ages = [5, 17, 22, 35, 45, 67, 90]
bins = [0, 12, 19, 35, 60, 100]
print(pd.cut(ages, bins=bins))
print()
print("age 35 ->", pd.cut([35], bins=bins)[0])
print("-> intervals are right-closed by default: (19, 35] includes 35.")

### Question 3

`0 -> nan`, `5 -> 'Child'`, `120 -> nan`. -> **2 of 3** values came back missing.

Both values outside the outermost edges became `NaN`, silently.

And `0` is the surprise: it is not beyond the range, it *is* the bottom
edge. But intervals are right-closed, so the first bin is `(0, 12]` — which
excludes 0. The lowest edge is never included in anything.

The deck says 'there is always one more edge than label' and stops. It does
not say that anything outside those edges is discarded without warning, and
that is the behaviour that costs people rows. Q4 shows it on real data.

In [ ]:
bins = [0, 12, 19, 35, 60, 100]
labels = ["Child", "Teen", "Young Adult", "Adult", "Senior"]
out = pd.cut([0, 5, 120], bins=bins, labels=labels)
for v, l in zip([0, 5, 120], out):
    print("  %3d -> %r" % (v, l))
print()
print("NaN count:", out.isna().sum(), "of", len(out))

### Question 4

`medium 359`, `large 276`, `small 227`, `huge 212`, **`NaN 19`**. -> `1074` binned, `19` unbinned, of `1093` rows. Those 19 orders are worth **`274216.24`**.

Nineteen orders exceeded the top edge of 10,000 and vanished from the
banding. Nothing raised.

The row count is the cheap check — 1,074 against 1,093 — and it is the one
nobody runs, because `value_counts()` without `dropna=False` prints four
tidy bands that sum to 1,074 and look complete.

The money makes it worse: those 19 orders carry 274,216 of the 1.6M total —
about **17% of revenue in 1.7% of rows**. A 'sales by band' report built on this
is missing a sixth of the business, and every band in it is correct.

In [ ]:
edges = [0, 100, 500, 2000, 10000]
names = ["small", "medium", "large", "huge"]
band = pd.cut(orders["Sales"], bins=edges, labels=names)
print(band.value_counts(dropna=False).to_string())
print()
print("binned:", band.notna().sum(), "| unbinned:", band.isna().sum(),
      "| rows:", len(orders))
print()
print("orders above the top edge:", (orders["Sales"] > 10000).sum())
print("their total Sales:", round(orders.loc[orders["Sales"] > 10000, "Sales"].sum(), 2))

### Question 5

Both `max()` and `float("inf")` as the top edge -> **0** unbinned. -> the lowest sale is `3.20` and no row is exactly `0`.

`float("inf")` is the safer of the two: `max()` works today and breaks the
day a larger order arrives, which is exactly when you would least notice.
An open-ended top bin cannot overflow.

The bottom edge is fine here only by luck — the smallest sale is 3.20, so
nothing sits at or below 0. Had any order been exactly 0 it would have
dropped out, for the reason in Q3. `-float("inf")` or `right=False` closes
that off.

In [ ]:
names = ["small", "medium", "large", "huge"]
a = pd.cut(orders["Sales"], bins=[0, 100, 500, 2000, orders["Sales"].max()], labels=names)
b = pd.cut(orders["Sales"], bins=[0, 100, 500, 2000, float("inf")], labels=names)
print("with max() as the top edge, unbinned:", a.isna().sum())
print("with inf as the top edge, unbinned:  ", b.isna().sum())
print()
print("but the lowest sale is %.2f, and the bottom edge is 0" % orders["Sales"].min())
print("rows exactly equal to 0:", (orders["Sales"] == 0).sum())

PART B — qcut: edges the data chooses

### Question 6

`Q1 274`, `Q2 273`, `Q3 273`, `Q4 273`. -> intervals `(3.199, 117.78]`, `(117.78, 404.91]`, and so on.

Near-perfect quarters, because `qcut` chose the edges to make them so.

Note how uneven the intervals are: the first quarter of orders spans about
115 currency units and the last spans thousands. That is the shape of the
data, and it is the information `cut` with round-number edges throws away.

Also note the lowest edge is `3.199`, nudged just below the minimum so the
smallest value is included — `qcut` handles the Q3 problem for you.

In [ ]:
q = pd.qcut(orders["Sales"], q=4, labels=["Q1", "Q2", "Q3", "Q4"])
print(q.value_counts().sort_index().to_string())
print()
edges = pd.qcut(orders["Sales"], q=4)
print("intervals chosen:")
for iv in edges.cat.categories:
    print("  ", iv)

### Question 7

`cut` -> `small 227`, `medium 359`, `large 276`, `huge 231`. `qcut` -> roughly `273` in each.

Two correct answers to different questions.

**`cut` tells you about the business**: the bands mean something before you
see the data, so 'we took 231 huge orders' is a statement anyone can act on,
and next quarter's number is comparable to this one.

**`qcut` tells you about the distribution**: the bands are defined by the
data, so 'Q4' means 'the top quarter of these orders' and nothing more.
Compare two quarters and the boundaries have moved underneath you, so a
customer can drop from Q4 to Q3 without their spending changing at all.

Use `cut` for reporting and `qcut` for within-dataset analysis. Never use
`qcut` bands as a metric that gets tracked over time.

In [ ]:
names = ["small", "medium", "large", "huge"]
by_cut = pd.cut(orders["Sales"], bins=[0, 100, 500, 2000, float("inf")], labels=names)
by_q = pd.qcut(orders["Sales"], q=4, labels=["Q1", "Q2", "Q3", "Q4"])

print("cut (business edges):")
print(by_cut.value_counts().reindex(names).to_string())
print()
print("qcut (equal-sized groups):")
print(by_q.value_counts().sort_index().to_string())

### Question 8

`small 227` orders, mean profit **`-21.09`** · `medium 359`, **`-28.56`** · `large 276`, `79.67` · `huge 231`, `917.20`.

**The two smallest bands lose money on average**, and together they are
586 of the 1,093 orders — more than half the file. Total profit is negative
for both: `-4788.13` and `-10254.64`.

That is the finding banding was for. It is invisible in the overall mean
profit, which is comfortably positive because the 231 huge orders contribute
`211873.36`. Group the same column four ways and the business splits into a
profitable tail and a loss-making majority.

`observed=True` matters when grouping on a categorical: without it, Pandas
keeps every defined category including ones with no rows, and you get
groups of zero length with `NaN` aggregates.

That is a deliberate feature — a full set of categories is useful for a
report that must show every band — and a surprise if you did not expect the
empty rows.

In [ ]:
work = orders.copy()
work["Band"] = pd.cut(work["Sales"], bins=[0, 100, 500, 2000, float("inf")],
                      labels=["small", "medium", "large", "huge"])
summary = work.groupby("Band", observed=True).agg(
    orders=("OrderID", "count"),
    total_profit=("Profit", "sum"),
    mean_profit=("Profit", "mean"),
).round(2)
print(summary.to_string())

### Question 9

Quartiles of `Quantity` -> `276`, `287`, `270`, `260` — not equal. -> `50` distinct values across 1,093 rows.

'Roughly equal' is doing real work here. `qcut` computes quantile
boundaries and then assigns values to them, and a value can only go in one
bin — so every row sharing a value at a boundary lands on the same side.

With only 50 distinct quantities across 1,093 rows, each value appears about
22 times, and the boundaries cannot split those groups. The result is off by
up to 27 rows per quarter.

The fewer distinct values a column has, the less `qcut` can honour its
promise — and Q10 is what happens when it cannot honour it at all.

In [ ]:
q = pd.qcut(orders["Quantity"], q=4)
print(q.value_counts().sort_index().to_string())
print()
print("distinct Quantity values:", orders["Quantity"].nunique())
print("most common:")
print(orders["Quantity"].value_counts().head().to_string())

### Question 10

`pd.qcut(s, q=4)` on `[1,1,1,1,1,2,3,100]` -> **raises** `ValueError: Bin edges must be unique: Index([1.0, 1.0, 1.0, 2.25, 100.0]...). You can drop duplicate edges by setting the 'duplicates' kwarg`.

Five of the eight values are `1`, so the 25th and 50th percentiles are both
`1.0` and three of the five edges collapse onto the same number. A bin from
1.0 to 1.0 cannot hold anything, so `qcut` refuses.

The message names the escape hatch, and it is worth being careful with it.
`duplicates="drop"` merges the collapsed edges and returns **fewer bins than
you asked for** — pass `q=4` and get 2, with no error. If your code then
assigns four labels it raises somewhere else; if it does not, you have a
'quartile' column with two values in it.

The underlying problem is a column too lumpy to quarter. Check
`nunique()` before quantile-binning anything.

In [ ]:
s = pd.Series([1, 1, 1, 1, 1, 2, 3, 100])
print("values:", s.tolist())
print("quantiles:", [s.quantile(x) for x in (0, 0.25, 0.5, 0.75, 1)])
print(pd.qcut(s, q=4))